phase 3: preprocessing

splitting data, extracting features, and encoding categoricals without data leakage.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# load data
print('--- loading data ---')
df = pd.read_csv('../data/raw/Train.csv')
print(f'raw shape: {df.shape}')

--- loading data ---
raw shape: (3146, 13)


In [2]:
# feature extraction and dropping
print('--- feature engineering ---')
df['deathdate'] = pd.to_datetime(df['deathdate'])
df['month'] = df['deathdate'].dt.month

cols_to_drop = ['ID', 'location', 'deathdate']
df = df.drop(columns=cols_to_drop)
print(f'features remaining: {df.columns.tolist()}')

--- feature engineering ---
features remaining: ['zone', 'gender', 'age', 'avg_temperature', 'max_temperature', 'min_temperature', 'precipitation', 'latitude', 'longitude', 'is_climate_sensitive', 'month']


In [3]:
# data splitting before encoding to prevent leakage
print('--- data splitting ---')
X = df.drop(columns=['is_climate_sensitive'])
y = df['is_climate_sensitive']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'train shape: {X_train.shape}')
print(f'val shape: {X_val.shape}')

--- data splitting ---
train shape: (2516, 10)
val shape: (630, 10)


In [4]:
# one-hot encoding
print('--- categorical encoding ---')
cat_cols = ['zone', 'gender', 'month']

# fit encoder on training data only
try:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
except TypeError:
    # fallback for older sklearn versions
    encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')

encoded_train = encoder.fit_transform(X_train[cat_cols])
encoded_val = encoder.transform(X_val[cat_cols])

encoded_cols = encoder.get_feature_names_out(cat_cols)

df_enc_train = pd.DataFrame(encoded_train, columns=encoded_cols, index=X_train.index)
df_enc_val = pd.DataFrame(encoded_val, columns=encoded_cols, index=X_val.index)

X_train = pd.concat([X_train.drop(columns=cat_cols), df_enc_train], axis=1)
X_val = pd.concat([X_val.drop(columns=cat_cols), df_enc_val], axis=1)

print(f'final train shape: {X_train.shape}')
print(f'final val shape: {X_val.shape}')

--- categorical encoding ---
final train shape: (2516, 23)
final val shape: (630, 23)


In [5]:
import os

# save processed data for next phase
print('--- saving processed data ---')
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)

print('data successfully saved to data/processed/')

--- saving processed data ---
data successfully saved to data/processed/


---
**phase 4 model selection strategy (based on preprocessing results):**
- preprocessing outcome: we successfully encoded categoricals and preserved the raw numericals, resulting in 23 distinct features.
- model choice 1: random forest classifier (serves as a highly robust baseline that resists overfitting on our small dataset).
- model choice 2: xgboost classifier (the gold standard for tabular data, perfectly handles our temperature multicollinearity without scaling).
- execution plan: train both ensemble models on the processed training set and evaluate their baseline performance on the validation set.